<a href="https://colab.research.google.com/github/spalominor/UWorks/blob/main/Taller3Criptografia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller #3: Cifrados de bloque y modos de operación


In [ ]:
# Instalar las dependencias necesarias
!pip install pycryptodome --quiet

# Importar las dependencias necesarias
import math
import random

# Importar un diccionario con valores por default
from collections import defaultdict

from Crypto.Cipher import AES, DES, DES3
from Crypto.Util.Padding import pad, unpad
from Crypto.Random import get_random_bytes

# Constante global del tamaño de bloque para AES
BLOCK_SIZE = 16

# Problema 1: Efecto mariposa criptográfico


Un desarrollador de software nota que, durante la transmisión de archivos cifrados a través de una red inalámbrica, los paquetes ocasionalmente sufren la alteración de un bit debido al ruido del canal. El desarrollador necesita decidir qué modo de operación de AES (CBC o CTR) es el más adecuado para minimizar el impacto destructivo del ruido sobre el archivo recuperado.

### Objetivos del ejercicio:

1. **Texto plano:** Defina una variable con un texto plano que contenga al menos 4 bloques completos (mínimo 64 bytes).
2. **Configuración de AES-CBC:** Aplique *padding* al texto plano, genere una llave aleatoria de 16 bytes y un vector de inicialización (IV) aleatorio de 16 bytes.
3. **Ruido en el canal:** Cifre el texto plano con la configuración anterior y modifique un bit aleatorio en el texto cifrado.
4. **Descifrado:** Descifre el texto corrupto.
5. **Repetición en AES-CTR:** Repita el experimento anterior utilizando el modo AES-CTR. Tenga en cuenta que la configuración de AES para este modo de operación es diferente.

### Preguntas:

1. Al imprimir el texto plano recuperado, ¿cuántos bytes y bloques se vieron alterados por la modificación en cada uno de los modos de uso? Explique detalladamente el comportamiento observado.
2. Si el canal de transmisión es altamente ruidoso y no cuenta con mecanismos de corrección de errores, ¿cuál modo de uso recomendaría implementar y por qué?

# Solución Problema #1

### Inicialización

In [ ]:
# Definir las variables iniciales
random.seed(23) # Para reproducir el experimento

# Definir el mensaje
message = "Mi abuelo siempre me decia, mucho cuidado con los de Cabo Verde. -La Cobra"
print(f"Mensaje: {message}")
print(f"Longitud del mensaje: {len(message)} bytes")
print(f"Bloques del mensaje: {math.ceil(len(message) / BLOCK_SIZE)}")

# Generación de las llaves
key = get_random_bytes(16)
iv_cbc = get_random_bytes(16)
nonce_ctr = get_random_bytes(8)

# Generar aleatoriamente el bit a cambiar
# Posición aleatoria del mensaje
byte_position = random.randint(0, len(message) - 1)
# Posición aleatoria del byte (8 bits)
bit_position = random.randint(0, 7)


def bit_flip(message: bytes, byte: int, bit: int):
    """
    Introduce un cambio de bit aleatorio en el mensaje.
    Args:
        message (bytes): El mensaje a modificar.
        byte_position (int): La posición del byte modificado.
        bit_position (int): La posición del bit modificado.
    Returns:
        bytes: El mensaje modificado.
    """
    corrupted_message = bytearray(message)

    # XOR para hacer el flip de bit en el byte elegido
    corrupted_message[byte_position] ^= (1 << bit_position)

    return bytes(corrupted_message)


def show_byte_error(byte_value: int, bit_position: int):
    """
    Imprime un circunflejo (^) debajo
    del byte que fue modificado.
    """
    # Calculamos la posición en la cadena
    print(" " * (byte_position * 2) + " " + "^")

Mensaje: Mi abuelo siempre me decia, mucho cuidado con los de Cabo Verde. -La Cobra
Longitud del mensaje: 74 bytes
Bloques del mensaje: 5


### Modo CBC

In [ ]:
# Definir el cifrador
cypher_cbc = AES.new(key, AES.MODE_CBC, iv=iv_cbc)

# Cifrar el mensaje original
message_padded = pad(message.encode(), BLOCK_SIZE)
message_encrypted_cbc = cypher_cbc.encrypt(message_padded)

# Introducir ruido en el mensaje cifrado
message_corrupted_cbc = bit_flip(message_encrypted_cbc,
                                 byte_position,
                                 bit_position)

# Descifrar el mensaje corrompido
decypher_cbc = AES.new(key, AES.MODE_CBC, iv=iv_cbc)
decrypted_message_cbc = decypher_cbc.decrypt(message_corrupted_cbc)

# Si el bit afectado fue uno perteneciente al padding, lanzará un error
try:
    decrypted_message_cbc = unpad(decrypted_message_cbc, BLOCK_SIZE)
except ValueError:
    print("El bit afectado pertenece al padding")
    decrypted_message_cbc = decrypted_message_cbc.decode('utf-8',
                                                         errors='replace')

# Imprimir los resultados
print(f"Mensaje original a cifrar: \n{message_padded}\n")

print(f"Mensaje cifrado: \n{message_encrypted_cbc.hex()}")
show_byte_error(byte_position, bit_position)

print(f"Mensaje corrompido: \n{message_corrupted_cbc.hex()}")
show_byte_error(byte_position, bit_position)
print(f"Ruido introducido en: Byte {byte_position}, bit {bit_position}")

print(f"\nMensaje descifrado: \n{decrypted_message_cbc}")

Mensaje original a cifrar: 
b'Mi abuelo siempre me decia, mucho cuidado con los de Cabo Verde. -La Cobra\x06\x06\x06\x06\x06\x06'

Mensaje cifrado: 
85ba1ce8fa0aa55f077e7c201870c1915ce186d6d457b7e4ded4a2a5c3be0fba0973adf90571f85f8c228fcebdfd6aa2026732219bc79f5c3445879c006b4657bb61d5fc24c5ce57895dc0f02b3ed364
                                                                           ^
Mensaje corrompido: 
85ba1ce8fa0aa55f077e7c201870c1915ce186d6d457b7e4ded4a2a5c3be0fba0973adf90573f85f8c228fcebdfd6aa2026732219bc79f5c3445879c006b4657bb61d5fc24c5ce57895dc0f02b3ed364
                                                                           ^
Ruido introducido en: Byte 37, bit 1

Mensaje descifrado: 
b'Mi abuelo siempre me decia, much|\xbf9%.f\xab\xe6\xf0\xb5\xd4T\x90\x08\x84\x98s de Aabo Verde. -La Cobra'


### Modo CTR

In [ ]:
# Definir el cifrador en modo COUNTER
cypher_ctr = AES.new(key, AES.MODE_CTR, nonce=nonce_ctr)

# Cifrar el mensaje original
# No se necesita padding, pero sí pasar a bytes
message_bytes = message.encode('utf-8')
message_encrypted_ctr = cypher_ctr.encrypt(message_bytes)

# Introducir ruido en el mensaje cifrado
message_corrupted_ctr = bit_flip(message_encrypted_ctr,
                                 byte_position,
                                 bit_position)

# Descifrar el mensaje corrompido
decypher_ctr = AES.new(key, AES.MODE_CTR, nonce=nonce_ctr)
decrypted_message_ctr = decypher_ctr.decrypt(message_corrupted_ctr)

# Imprimir los resultados
print(f"Mensaje original a cifrar: \n{message_bytes}\n")

print(f"Mensaje cifrado: \n{message_encrypted_ctr.hex()}")
show_byte_error(byte_position, bit_position)

print(f"Mensaje corrompido: \n{message_corrupted_ctr.hex()}")
show_byte_error(byte_position, bit_position)
print(f"Ruido introducido en: Byte {byte_position}, bit {bit_position}")

print(f"\nMensaje descifrado: \n{decrypted_message_ctr}")

Mensaje original a cifrar: 
b'Mi abuelo siempre me decia, mucho cuidado con los de Cabo Verde. -La Cobra'

Mensaje cifrado: 
5a8d3fedacdcdddfdd90a1e770e80a46c62e445fe1d71d8de4d723b7b882595f6fdf24c493d28738fefb9f3cd997bc7ab4dd7df9a24fde013df84ea5db0f9e5e5e279e5133ac9d77075f
                                                                           ^
Mensaje corrompido: 
5a8d3fedacdcdddfdd90a1e770e80a46c62e445fe1d71d8de4d723b7b882595f6fdf24c493d08738fefb9f3cd997bc7ab4dd7df9a24fde013df84ea5db0f9e5e5e279e5133ac9d77075f
                                                                           ^
Ruido introducido en: Byte 37, bit 1

Mensaje descifrado: 
b'Mi abuelo siempre me decia, mucho cuifado con los de Cabo Verde. -La Cobra'


### Conclusión

**1. Al imprimir el texto plano recuperado, ¿cuántos bytes y bloques se vieron alterados por la modificación en cada uno de los modos de uso? Explique detalladamente el comportamiento observado.**

**Modo CBC**

AES opera sobre bloques de 16 bytes. El mensaje utilizado tenía 74 bytes, por lo que se dividió en 5 bloques después de aplicar padding.

En el experimento afectamos el bit 1 del byte 37. Este corresponde al bloque 3 del mensaje cifrado.

$$P_1 = D_K(C_1) \oplus IV$$
$$P_2 = D_K(C_2) \oplus C_1$$
$$P_3 = D_K(C_3*) \oplus C_2$$
$$P_4 = D_K(C_4) \oplus C_3*$$
$$P_5 = D_K(C_5) \oplus C_4$$

Viendo el proceso de descifrado, podemos observar que:
Al alterar el bloque de texto cifrado $C3*$, este entra en la función de descifrado de AES, $D_k$, debido al efecto avalancha provocado por las funciones internas de AES en cada ronda (SubBytes, ShiftRows, MixColumns), todo el bloque de texto plano $P_3$ se verá afectado, y al hacer XOR con $C_2$, al final todo parece algo completamente aleatorio.

Además, el bloque de texto plano $P_4$, se ve afectado un poco. Después de calcular el $D_k(P_4)$, se hace XOR con el $C3*$, por lo que el bit cambiado, afecta al byte que hace XOR. En este caso se afectaría quirúrgicamente el byte 5 (37 mod 16 = 5, dado que son bloques de 16).

Al final, son afectados 17 bloques, 16 del bloque $P_3$, y 1 del bloque $P_4$. Claramente se ve en el resultado, los dos primeros bloques de 16 correctos, el tercer bloque corrupto, y el quinto byte del cuarto bloque afectado:


```
Bloque 0 (00-15):  M i   a b u e l o   s i e m p r
Bloque 1 (16-31):  e   m e   d e c i a ,   m u c h
Bloque 2 (32-47):  \x18 * \xa4 \xca \x92 X 6 j _ [ p \x81 ( \x0c \x84 k
Bloque 3 (48-63):  s   d e   A a b o   V e r d e .
                   | | | | | |
                   0 1 2 3 4 5  <-- Byte 5 del bloque (37 mod 16)
```
**Modo CTR**

En CTR, AES genera un keystream que se combina con el texto mediante la operación XOR. Por ello, un bit alterado en el texto cifrado modifica únicamente ese mismo bit del texto plano recuperado. No existe propagación del error hacia otros bloques, por lo que el daño permanece completamente localizado.

```

Mensaje descifrado:
b'Mi abuelo siempre me decia, mucho cuifado con los de Cabo Verde. -La Cobra'
                                       ^
```

**2. Si el canal de transmisión es altamente ruidoso y no cuenta con mecanismos de corrección de errores, ¿cuál modo de uso recomendaría implementar y por qué?**

Se recomienda utilizar AES-CTR, ya que los errores producidos por el ruido no se propagan. Un cambio en un único bit del texto cifrado afecta únicamente ese mismo bit del texto plano, preservando el resto del mensaje.

Por otro lado, AES-CBC propaga el error al bloque completo correspondiente y altera además un bit del bloque siguiente, lo que incrementa significativamente la corrupción del mensaje.

# Problema 2: Error de capa 8

Un desarrollador de una aplicación bancaria decidió simplificar el código de comunicación fijando el IV de AES-CBC como una constante llena de bytes en cero (b'\x00' * 16), bajo la premisa de que "mientras la llave sea secreta y aleatoria, el sistema es impenetrable". En este sistema, todas las transacciones de la sesión se cifran utilizando la misma llave secreta, pero cometiendo el error de reutilizar ese mismo IV estático de ceros para cada mensaje.

Usted (Trudy) ha interceptado una ráfaga de transacciones en la red inalámbrica del banco. Aunque no puede ver el contenido del texto plano, su misión es construir un script de criptoanálisis para agrupar e identificar cuáles transacciones corresponden a comandos idénticos, demostrando la vulnerabilidad del sistema.

Se sabe que cada transacción inicia siempre con una palabra de exactamente 16 bytes (16 caracteres) que indica el tipo de operación. Existen cuatro tipos de transacciones posibles en el sistema:

- TX_TYPE:BALANCE_
- TX_TYPE:TRANSFER
- TX_TYPE:WITHDRAW
- TX_TYPE:BLOCKED_

### Objetivos del ejercicio:

1. Escriba un programa que agrupe automáticamente cuáles mensajes pertenecen a la misma operación bancaria. Indique explícitamente los índices de los mensajes que colisionan.

### Preguntas:

*Nota de contexto: En este escenario, un IV estático significa que se utiliza exactamente el mismo Vector de Inicialización en todos los mensajes transmitidos.*

1. ¿Cuál es la debilidad de implementar un IV constante con el valor cero?
2. ¿Esta debilidad seguiría presente si el IV se mantiene constante pero se genera inicialmente de forma aleatoria (por ejemplo, usando una palabra fija)?
3. ¿Esta vulnerabilidad por sí sola permite al atacante extraer la llave secreta o leer el contenido del resto del texto plano?

In [ ]:
mensajes_cifrados_interceptados = [
    "24faaf01885ebd155554f050a3d4521db89019f1913f620214c392d398dbc062348a605e4aaa7c8355465757e6ff97cc", # Mensaje 0
    "9911ab4ea4c273719602169fb003134f83d918211d30b09fc2cabc2a22e9ce2b8b024311297b5b097b114751393fd731", # Mensaje 1
    "24faaf01885ebd155554f050a3d4521d24da6ce925122c9868e630699b0fde9d10be994785263d797e94191170c183f2", # Mensaje 2
    "4b599741e536bf5f3ff11350501d7239dbd81675e48b4a0c560b0cbd21650d64b77742d54a62753a3d669c7c82ccc404", # Mensaje 3
    "9911ab4ea4c273719602169fb003134fbc8be26b6da1a0a3a02e4e0a33a86306796196d6a0a3e997d105ec79cb749ae7", # Mensaje 4
    "c6d9ffa78f2605d507ca37638f6e904f17bbb907ed63bf5ccbe1ccc4c3ce91a6799b9de5960d4290aed5c7839ee08876", # Mensaje 5
    "24faaf01885ebd155554f050a3d4521d4f8c9b879b7331eb8ca90dd3c16f44ab2bdc64aab3ed00ef37a4d699573c8d96", # Mensaje 6
    "4b599741e536bf5f3ff11350501d72396797617e4053fc5ee1d3ad76a3ca5ba6b82c8c6a55e05a6cd9bd930a6b5f35c6", # Mensaje 7
    "9911ab4ea4c273719602169fb003134f83b7fc334dc59d10e122499ff69662d6a3e1787125c6a6f206507cb3469e650d", # Mensaje 8
    "c6d9ffa78f2605d507ca37638f6e904f341e8a9a987d83c3c88a129344cad5c41a21f596aa9256425d8e5565b2da4948", # Mensaje 9
]

# Solución Problema #2

## Implementación

In [ ]:
# Crear un diccionario para guardar las apariciones
frequencies = defaultdict(list)

# Iterar por cada mensaje interceptado
for index, message in enumerate(mensajes_cifrados_interceptados):
    # Extraer el primer bloque cifrado (16 bytes = 32 carácteres en hex)
    command = message[:32]

    # Añadir el índice del mensaje según con qué comando colisionó
    # Clave: 16 bytes del comando - Valor: Índice
    frequencies[command].append(index)


# Imprimir los resultados
for command, appearances in frequencies.items():
    # Imprimir el primer bloque (Clave)
    print(f"{command}")
    print(f"Apariciones: {len(appearances)} -> {appearances}\n")

# Construir una nueva lista con los mensajes ordenados
ordered_messages = []
for appearances in frequencies.values():
    for index in appearances:
        message = mensajes_cifrados_interceptados[index]

        # Ingresar un espacio en blanco a partir del byte 16
        message = message[:32] + " | " + message[32:]

        # Agregar el mensaje a la lista
        ordered_messages.append(message)

print(f"Mensajes organizados:")
print(f"{"\n".join(ordered_messages)}")

24faaf01885ebd155554f050a3d4521d
Apariciones: 3 -> [0, 2, 6]

9911ab4ea4c273719602169fb003134f
Apariciones: 3 -> [1, 4, 8]

4b599741e536bf5f3ff11350501d7239
Apariciones: 2 -> [3, 7]

c6d9ffa78f2605d507ca37638f6e904f
Apariciones: 2 -> [5, 9]

Mensajes organizados:
24faaf01885ebd155554f050a3d4521d | b89019f1913f620214c392d398dbc062348a605e4aaa7c8355465757e6ff97cc
24faaf01885ebd155554f050a3d4521d | 24da6ce925122c9868e630699b0fde9d10be994785263d797e94191170c183f2
24faaf01885ebd155554f050a3d4521d | 4f8c9b879b7331eb8ca90dd3c16f44ab2bdc64aab3ed00ef37a4d699573c8d96
9911ab4ea4c273719602169fb003134f | 83d918211d30b09fc2cabc2a22e9ce2b8b024311297b5b097b114751393fd731
9911ab4ea4c273719602169fb003134f | bc8be26b6da1a0a3a02e4e0a33a86306796196d6a0a3e997d105ec79cb749ae7
9911ab4ea4c273719602169fb003134f | 83b7fc334dc59d10e122499ff69662d6a3e1787125c6a6f206507cb3469e650d
4b599741e536bf5f3ff11350501d7239 | dbd81675e48b4a0c560b0cbd21650d64b77742d54a62753a3d669c7c82ccc404
4b599741e536bf5f3ff11350501d7239 | 6

## Conclusión



**1. ¿Cuál es la debilidad de implementar un IV constante con el valor cero?**

La principal debilidad de esta implementación es **usar un IV constante. Teniendo en cuenta la manera en la que AES en modo CBC cifra el mensaje, podemos ir a la fórmula del primer bloque:

$$C_1 = E_k(P_1⊕IV)$$

Entonces tenemos el primer bloque de texto plano que se repite cada tanto, misma llave para cifrar todos los mensajes, y un IV constante; unido a que la función de encripción de AES es determinista, podemos obtener mismos mensajes cifrados bajo las mismas entradas.

La debilidad radica en eliminar la fuente de aleatoriedad que proporciona el IV al algoritmo determinista de AES, **el primer bloque será vulnerable mientras se reutilice el IV** bajo los mismos textos planos.

**2. ¿Esta debilidad seguiría presente si el IV se mantiene constante pero se genera inicialmente de forma aleatoria (por ejemplo, usando una palabra fija)?**

Sí, la debilidad seguiría existiendo. En caso de que se generara un IV criptográficamente aleatorio, robusto, caótico, impredecible, cuando el IV se reutiliza, pierde todas sus propiedades. Veámoslo:

$$ X = P_1 ⊕ IV $$
$$ C_1 = E_k(X) $$

Cuando se reutiliza el IV, mismos textos planos producirán mismas entradas X para el algoritmo determinista de AES. El cifrado del primer bloque seguirá siendo determinista, y bajo mensajes con el mismo bloque de texto plano, devolverá el mismo texto cifrado.

**3. ¿Esta vulnerabilidad por sí sola permite al atacante extraer la llave secreta o leer el contenido del resto del texto plano?**

No, esta vulnerabilidad por sí sola no permite extraer la llave ni el resto del texto plano.

A priori, da información de cuando dos mensajes empiezan igual y agrupar mensajes. Sin embargo, bajo esta implementación, se podrían realizar otros ataques que requieran un poco más de esfuerzo.

- Se podría plantear un **ataque CPA (Chosen-Plaintext-Attack)**, si digamos tengo acceso al cifrador que usa la aplicación bancaria, puedo preparar listas de mensajes verosimiles en base a la lista de comandos posibles. Dado que tengo el IV, y la llave ya está implícita en el cifrador, cuando obtenga un mensaje cifrado igual, sé que encontré el texto plano que lo generó en primer momento.

- Gracias a los mensajes interceptados conozco: el primer bloque cifrado, el IV, y los cuatro comandos en texto plano, es decir, tengo un par valor de entrada y valor de salida, se puede plantear algo así como **un ataque KPA (Known-Plaintext Attack)**. Si por otra razón, el desarrollador no usó una llave generada correctamente, o sea, no es aleatoria, y se reduce en gran cantidad el espacio de llaves, solo bastaría con encriptar el mismo primer bloque, hasta obtener la llave que genere el texto cifrado interceptado, y así poder leer todo el mensaje.

Sin embargo, ya son muchos errores de capa 8.

# Problema 3: El oráculo involuntario

Tras el desastre de seguridad del ejercicio anterior (donde el uso de un $IV$ fijo lleno de ceros permitió filtrar los tipos de transacciones), el desarrollador de la aplicación bancaria aseguró haber "aprendido la lección sobre mantener los $IV$ estáticos". Su nueva y brillante propuesta para el sistema de mensajería fue: "Si el problema es que el $IV$ no cambia y es predecible, usemos la propia llave secreta y aleatoria como Vector de Inicialización ($IV=K$). Como la llave cambia en cada nueva sesión y es un secreto absoluto, el $IV$ ahora será robusto, variable e imposible de adivinar por un atacante, ahorrándonos líneas de código en el proceso".

Usted (Trudy) ha logrado montar un ataque activo de texto cifrado escogido (CCA) sobre la API del servidor. Aunque no conoce la llave del sistema, dispone de una función para enviar cualquier texto cifrado que usted desee (en formato hexadecimal) y el servidor le devolverá una confirmación o, en su defecto, un mensaje de error con el volcado hexadecimal de los datos dañados si el texto plano resultante no se puede leer como texto ASCII válido.

### Objetivos del ejercicio:

1. Utilice las funciones `encrypt` y `decrypt` (modelo CCA) para realizar un ataque activo y extraer la llave secreta $K$ del servidor.

### Pistas:

1. **El Oráculo de Error:** Observe detenidamente el comportamiento de la función `decrypt`. Cuando esta función descifra un texto cifrado cuyo resultado no corresponde a un texto ASCII válido, el servidor no se limita a denegar el acceso, sino que expone el texto plano completo en formato hexadecimal dentro del mensaje de error.
2. **La Inyección Estructurada:** Tome un bloque $C$ de texto cifrado legítimo (de 16 bytes). Ahora, construya un texto cifrado malicioso uniendo tres bloques exactos: $C \ || \ \emptyset \ || \ C$ (donde $\emptyset$ es un bloque de 16 bytes lleno de ceros). ¿Qué ocurre matemáticamente bloque por bloque al descifrar esta estructura bajo el modo CBC si recordamos que $IV = K$?

In [ ]:
# Configuración del servidor (La llave es secreta y actúa también como IV)
KEY = get_random_bytes(16)

In [ ]:
def encrypt(plaintext_hex):
    """Cifra un mensaje en formato hexadecimal usando IV = KEY."""
    try:
        plaintext = bytes.fromhex(plaintext_hex)
    except ValueError:
        return {"error": "Invalid hex format"}

    if len(plaintext) % 16 != 0:
        return {"error": "Data length must be multiple of 16"}

    cipher = AES.new(KEY, AES.MODE_CBC, KEY) # El "nuevo" error crítico: IV = KEY
    encrypted = cipher.encrypt(plaintext)
    return {"ciphertext": encrypted.hex()}


def decrypt(ciphertext_hex):
    """Descifra el texto cifrado y actúa como un Oráculo."""
    try:
        ciphertext = bytes.fromhex(ciphertext_hex)
    except ValueError:
        return {"error": "Invalid hex format"}

    if len(ciphertext) % 16 != 0:
        return {"error": "Data length must be multiple of 16"}

    cipher = AES.new(KEY, AES.MODE_CBC, KEY)
    decrypted = cipher.decrypt(ciphertext)

    try:
        # Intenta asegurar que el texto plano sea ASCII válido
        decrypted.decode()
    except UnicodeDecodeError:
        # Si falla la decodificación, el servidor expone el texto plano en HEX
        return {"error": f"Invalid plaintext: {decrypted.hex()}"}

    return {"success": "Your message has been received"}

# Solución Problema #3

## Implementación

Idea:

Cifrar un mensaje de longitud un bloque.

$$ C_1 = E_k(P_1⊕IV) = E_k(P_1⊕K) $$

Crear un mensaje cifrado inteligente, con la siguiente concatenación:

$$ C = C_1 + x00 * 16 + C_1 $$

Desciframos

$$ P_1 = D_k(C_1)⊕IV $$
$$ P_1 = D_k(C_1)⊕K $$

Vemos una relación directa que expone la llave. Dadas las propiedades del XOR, requerimos el valor de $D_k(C_1)$ para despejar la llave $K$.

$$ P_2 = D_k(x00 * 16) ⊕ C_1 $$
$$ P_3 = D_k(C_3) ⊕ (x00 * 16) $$
$$ P_3 = D_k(C_1) $$

Tenemos la suficiente información para realizar el despeje:

$$ K = D_k(C_1) ⊕ K ⊕ D_k(C_1) $$
$$ K = P_1 ⊕ P_3 $$

In [ ]:
# Elegimos un texto de 16 bytes a encriptar
p1_text = "TX_TYPE:BALANCE_"
p1_hex = p1_text.encode().hex()

# Obtenemos su encripción
c1 = encrypt(p1_hex)
c1_hex = c1["ciphertext"]
print("Bloque de texto plano cifrado:")
print(c1_hex)

# Elegimos un texto cifrado a descifrar que nos brinde información
# 16 bytes en cero \0x00*16 para el segundo bloque
c2_text = 0
c2_hex = c2_text.to_bytes(16, 'big').hex()

# Concatenamos todo, primer y último bloque iguales
chosen_ciphertext = c1_hex + c2_hex + c1_hex

# Desciframos el mensaje elegido
plaintext = decrypt(chosen_ciphertext)

# Extraemos el hex devuelto
plaintext_hex = plaintext["error"]
plaintext_hex = plaintext_hex[19:]
print("\nTexto cifrado elegido descifrado")
print(plaintext_hex)

# Dividimos los 3 bloques
block_1 = plaintext_hex[:32]
block_2 = plaintext_hex[32:64]
block_3 = plaintext_hex[64:]

# Despejamos haciendo XOR
key_found_int = int(block_1, 16) ^ int(block_3, 16)
key_found_bytes = key_found_int.to_bytes(16, byteorder='big')

# Resultados
print(f"\nLa llave encontrada en el ataque: {key_found_bytes}")
print(f"La llave usada en el sistema es:  {KEY}")

# Creamos un nuevo descifrador con la misma arquitectura para validar
new_decypher = AES.new(key_found_bytes, AES.MODE_CBC, key_found_bytes)

# Mensaje 100% real interceptado de la aplicación bancaria
message_sent = "TX_TYPE:TRANSFER_FROM:001-MERINO_TO:002-PALOMINO_AMOUNT:$9999999"
message_sent_hex = message_sent.encode().hex()

# Usamos el Oráculo
ciphertext_intercepted_hex = encrypt(message_sent_hex)["ciphertext"]
ciphertext_intercepted = bytes.fromhex(ciphertext_intercepted_hex)

# Obtenemos el texto plano con la llave encontrada
plaintext_intercepted = new_decypher.decrypt(ciphertext_intercepted)
print("\n\nPrueba")
print("El mensaje de real interceptado fue:")
print(plaintext_intercepted.decode("utf-8", errors="ignore"))

Bloque de texto plano cifrado:
b22e072b9ba05057824864baa36c2375

Texto cifrado elegido descifrado
54585f545950453a42414c414e43455f3d3c8ce06dbd538c0ef66c92b1d1742530f91829cb7f82964078f73d207affc5

La llave encontrada en el ataque: b'd\xa1G}\x92/\xc7\xac\x029\xbb|n9\xba\x9a'
La llave usada en el sistema es:  b'd\xa1G}\x92/\xc7\xac\x029\xbb|n9\xba\x9a'


Prueba
El mensaje de real interceptado fue:
TX_TYPE:TRANSFER_FROM:001-MERINO_TO:002-PALOMINO_AMOUNT:$9999999


# Prolema 4: No todas las llaves son buenas - Involution Attack

Triple DES no cifra tres veces consecutivas. Por razones de compatibilidad con el DES tradicional, se diseñó utilizando un esquema de tres capas llamado EDE (Encrypt Decrypt Encrypt).

La librería PyCryptodome permite pasar llaves de 16 bytes (128 bits) o de 24 bytes (192 bits). En este caso, si le pasamos una llave de 16 bytes, la librería implementa lo que técnicamente se conoce como 3DES Opción 2:

- Divide los 16 bytes en dos llaves DES de 8 bytes cada una: $K_1$​ y $K_2$​.
- Automáticamente define que la tercera llave es igual a la primera ($K3​=K1$​).

Por lo tanto, la ecuación interna de la librería para cifrar un bloque con una llave de 16 bytes es:

$$C = E_{K_1}(D_{K_2}(E_{K_1}(P)))$$

**El problema ...**

DES tiene 4 llaves conocidas como "llaves débiles". Su particularidad matemática es que sus subllaves internas de ronda son simétricas, lo que provoca que cifrar sea exactamente lo mismo que descifrar. Si $K$ es una llave debil entonces:

$$  E_K(X) = D_K(X)  $$

Equivalentemente,

$$  E_K(E_k(X)) = X  $$

Las 4 llaves debiles de DES son (en hexadecimal):

- 0101010101010101
- 1F1F1F1F0E0E0E0E
- FEFEFEFEFEFEFEFE
- E0E0E0E0F1F1F1F1

**Nota:** A pesar de que teóricamente DES utiliza una llave de 56 bits, las implementaciones prácticas exigen una longitud de 64 bits (8 bytes). Esto ocurre porque el algoritmo ignora por completo el último bit de cada byte (el bit menos significativo). Históricamente, este octavo bit se utilizaba como un "bit de paridad" para verificar, a nivel de hardware antiguo, que la llave no se hubiera corrompido durante su transmisión.

Este ejercicio busca ilustrar la existencia de llaves debiles en DES y el modelo de ataque Chosen-Key Attack.

### Objetivos del ejercicio:
Para el desarrollo de este ejercicio, utilice las implementaciones de DES y 3DES que provee la librería PyCryptodome.

1. Verifique experimentalmente que se cumple la propiedad descrita para las llaves débiles de DES, es decir, muestre que al cifrar un mensaje cualquiera con una de estas llaves, la operación de cifrado se comporta de manera idéntica a la de descifrado.
2. Construya una llave débil válida para 3DES a partir de la combinación de dos llaves débiles distintas de DES.
3. Utilice la función `get_ciphertext` enviando la llave débil que construyó para capturar el texto cirado del mensaje secreto del sistema.
4. Diseñe un script que tome dicho texto cifrado y a traves de la función  `encrypt` realice el post-procesamiento necesario para extraer el texto plano secreto.

### Preguntas:

1. Explique matemáticamente por qué el utilizar llaves débiles de DES colapsa por completo la estructura de capas EDE (Encrypt-Decrypt-Encrypt) de Triple DES.

In [ ]:
# Parámetros internos del sistema (datos secretos)
_IV_SECRETO = get_random_bytes(8)
_TEXTO_PLANO_SECRETO = f"TOKEN-SESION-{get_random_bytes(6).hex().upper()}"

def _xor(a, b):
    return bytes(x ^ y for x, y in zip(a, b * (1 + len(a) // len(b))))

def encrypt(key_hex, plaintext_hex):
    """
    Simula el oráculo general de cifrado.
    Permite cifrar cualquier texto plano usando una llave provista.
    """
    try:
        key = bytes.fromhex(key_hex)
        plaintext = bytes.fromhex(plaintext_hex)

        """"
        El ECB no admite el uso de IVs pero el desarollador intento mejorar
        este modo incorporando un IV y aplicando XOR con el texto plano antes
        de cifrar y despues del cifrado.

        C = E_k(P XOR IV) XOR IV

        (Esta no es la debilidad fundamental en el esquema)
        """
        plaintext_masked = _xor(plaintext, _IV_SECRETO)

        cipher = DES3.new(key, DES3.MODE_ECB)
        ciphertext = cipher.encrypt(plaintext_masked)

        ciphertext_final = _xor(ciphertext, _IV_SECRETO)

        return {"ciphertext": ciphertext_final.hex()}
    except ValueError as e:
        return {"error": str(e)}

def get_ciphertext(key_hex):
    """
    Cifra el mensaje secreto del sistema utilizando la llave provista.
    """
    texto_plano_padded = pad(_TEXTO_PLANO_SECRETO.encode(), 8).hex()
    return encrypt(key_hex, texto_plano_padded)

In [ ]:
WEAK_KEYS = [
    "0101010101010101",
    "1F1F1F1F0E0E0E0E", # Llave débil corregida del enunciado original
    "FEFEFEFEFEFEFEFE",
    "E0E0E0E0F1F1F1F1"
]

dummy = b'HiWorld!'

### DES

In [ ]:
# Inicializar la tabla
results = []

# Crear una tabla para verificar la propiedad de las llaves débiles para DES
for weak_key in WEAK_KEYS:
    # Cifrar
    weak_key_bytes = bytes.fromhex(weak_key)
    cipher = DES.new(weak_key_bytes, DES.MODE_ECB)
    encrypted = cipher.encrypt(dummy)

    # Volver a cifrar (verificar el descifrado)
    decrypted = cipher.encrypt(encrypted)

    # Guardar los resultados
    results.append((weak_key, encrypted, decrypted))

# Imprimir los resultados en una tabla
print("Llave Débil\tEncripción\t2da Encripción")
for weak_key, encrypted, decrypted in results:
    print(f"{weak_key}\t{encrypted.hex()}\t{decrypted.hex()}")


Llave Débil	Encripción	2da Encripción
0101010101010101	ce8c97adee21060b	4869576f726c6421
1F1F1F1F0E0E0E0E	9a46883320643651	4869576f726c6421
FEFEFEFEFEFEFEFE	6110a15827ee5984	4869576f726c6421
E0E0E0E0F1F1F1F1	596ed045aa804cdf	4869576f726c6421


### 3DES

In [ ]:
# Definir dos llaves débiles
key_1, key_2 = WEAK_KEYS[0], WEAK_KEYS[1]
weak_key = key_1 + key_2

# Capturamos un texto cifrado con la llave creada
intercepted = get_ciphertext(weak_key)['ciphertext']
print(f"Texto cifrado interceptado: {intercepted}")

# Encriptamos el texto cifrado interceptado
cka_result = encrypt(weak_key, intercepted)['ciphertext']
print(f"Texto plano recuperado: {cka_result}")

# Obtenemos el string a partir del hex
cka_result_str = bytes.fromhex(cka_result).decode()

# Verificar si coincide con el secreto global
print(f"\nTexto plano recuperado: {cka_result_str}")
print(f"Texto plano secreto:    {_TEXTO_PLANO_SECRETO}")



Texto cifrado interceptado: e9c3caa2ed9fa6725560e41533a7e459c8521c34c482ab7327256fcf1293e172
Texto plano recuperado: 544f4b454e2d534553494f4e2d43414238423644424339393807070707070707

Texto plano recuperado: TOKEN-SESION-CAB8B6DBC998
Texto plano secreto:    TOKEN-SESION-CAB8B6DBC998


## Conclusión

¿Por qué matemáticamente estas llaves rompen el esquema?

Triple DES con una llave de 16 bytes implementa el esquema:

$$C = E_{K_1}(D_{K_2}(E_{K_1}(P)))$$

Si $K_1$ y $K_2$ son llaves débiles de DES, entonces para ambas se cumple:

$$E_{K_i} = D_{K_i}$$

Sustituyendo esta propiedad en la expresión anterior se obtiene:

$$C = E_{K_1}(E_{K_2}(E_{K_1}(P)))$$

por lo que desaparece completamente la capa de descifrado (*Decrypt*) que caracteriza la estructura EDE. Además, como una llave débil satisface:

$$E_K(E_K(X)) = X$$

la operación de cifrado también actúa como su propia inversa. En consecuencia, el texto plano puede recuperarse aplicando nuevamente la operación de cifrado con la misma combinación de llaves, anulando el propósito de la construcción EDE y colapsando la seguridad esperada de Triple DES.

# Problema 5: Padding Oracle Attack

Un sistema de autenticación cifra un token de sesión aleatorio utilizando AES-128 en modo CBC. El desarrollador implementó una función de validación que descifra los tokens enviados por los usuarios y verifica si el formato del relleno es correcto. Aunque el sistema nunca muestra el contenido del texto plano si ocurre un error, devuelve un valor booleano indicando si el relleno fue válido o no.

### Objetivos del ejercicio:

1.  Utilice la función `get_ciphertext` para obtener el texto cifrado del mensaje secreto.

2. Usando el oraculo de relleno (función `verificar_relleno_oracle`) obetener el mensaje secreto en texto plano.

### Idea del ataque:

**Maleabilidad:** Recordar como funciona el descifrado en el modo CBC. A modo de convención, en esta implementación, cuando se envia un mensaje se hace en el siguiente formato:
<p></p><p></p>
[ IV (16 bytes) ] + [ Bloque CT 1 (16 bytes) ] + [ Bloque CT 2 (16 bytes) ] ...
<p></p><p></p>

Es decir, primer se envia el IV y luego el mensaje cifrado.

**Oráculo:**  Para descifrar un bloque de interés de 16 bytes (llamémoslo C_objetivo), debes construir un payload específico de exactamente 32 bytes estructurado así:
<p></p>
Payload = IV_falso (16 bytes) + C_objetivo  (16 bytes)
<p></p>

Al pasarle este paquete al método `verificar_relleno_oracle`, el servidor aislará esos 32 bytes, usará tu IV_falso para la operación XOR y validará el relleno únicamente sobre ese bloque.

El IV_falso es un arreglo de 16 bytes generado por ti que inicialmente puedes rellenar con ceros [0, 0, ..., 0]. Tu objetivo es realizar fuerza bruta únicamente sobre la posición del byte que estás atacando (probando valores del 0 al 255).
Cuando el oráculo te devuelva True por primera vez atacando la última posición (índice 15), significará que tu **byte_intento** logró que el último byte del texto plano descifrado por el servidor  se convirtiera exactamente en el byte 0x01 (un padding válido de 1 byte).

 En el momento en que el oráculo da True, se cumple la siguiente relación matemática en el servidor para esa posición específica:
<p></p>

 Valor_Intermedio = byte_intento $\oplus$ Padding_Buscado

<p></p>

 Una vez que calculas ese Valor_Intermedio (que representa la salida cruda de la caja negra de AES antes del XOR), puedes descubrir el carácter real del token haciendo un XOR final con el bloque original que hacía de IV en el texto cifrado interceptado:
<p></p>

 Texto_Plano_Real = Valor_Intermedio $\oplus$ IV_Real_Original

<p></p>

 Tras descubrir la posición 15, querrás descubrir la posición 14. Para ello, el oráculo ahora necesitará ver un padding válido de dos bytes: 0x02 0x02.

 El texto cifrado completo mide más de un bloque. Debes atacar los bloques de izquierda a derecha. Recuerda que el "IV Real" del bloque 1 es el IV original del sistema, pero el "IV Real" necesario para descifrar el bloque 2 es, de hecho, el bloque 1 del texto cifrado original.

In [ ]:
# Variables globales que simulan el secreto interno
# El mensaje secreto es un token aleatorio en formato hex de 16 bytes (32 caracteres ASCII)
_mensaje_secreto = get_random_bytes(16).hex()
#_mensaje_secreto = "SAMUELPALOMINO23".encode('utf-8').hex()
_llave_oculta = get_random_bytes(16)

def get_ciphertext():
    """Genera el texto cifrado correspondiente al mensaje secreto"""
    iv = get_random_bytes(16)
    cipher = AES.new(_llave_oculta, AES.MODE_CBC, iv=iv)

    # Al ser 32 bytes (múltiplo de 16), añadirá un 3er bloque completo de relleno (\x10 * 16)
    datos_con_padding = pad(_mensaje_secreto.encode("ascii"), 16)

    ct = cipher.encrypt(datos_con_padding)
    return {"ct": (iv + ct).hex()}

def verificar_relleno_oracle(texto_cifrado_hex):
    """
    Padding Oracle. Descifra el texto cifrado y verifica si el formato de padding es válido.

    True: Padding valido
    False: Padding invalido

    """
    try:
        datos_bytes = bytes.fromhex(texto_cifrado_hex)
        if len(datos_bytes) < 32:
            return {"result": False}

        # Convención estándar: IV es el primer bloque del mensaje
        # [ IV (16 bytes) ] + [ Bloque CT 1 (16 bytes) ] + [ Bloque CT 2 (16 bytes) ] ...
        iv = datos_bytes[:16]
        ct = datos_bytes[16:]

        cipher = AES.new(_llave_oculta, AES.MODE_CBC, iv=iv)
        pt_decrypted = cipher.decrypt(ct)

        # El oráculo intenta quitar el padding y revela si la estructura es correcta
        unpad(pt_decrypted, 16)
        return {"result": True}
    except ValueError:
        return {"result": False}

# Solución Problema #5

## Implementación

In [ ]:
BLOCK_SIZE = 16

def padding_oracle_attack():
    """
    Recupera el texto plano mediante un Padding Oracle Attack sobre AES-CBC.

    La función obtiene un texto cifrado desde el servidor y descifra cada uno de
    sus bloques utilizando un oráculo de validación de padding. Para ello,
    construye un IV falso, modifica sus bytes mediante fuerza bruta y aprovecha
    las respuestas del oráculo para recuperar el valor intermedio del descifrado.
    Posteriormente, reconstruye el texto plano aplicando la operación XOR con el
    bloque anterior del ciphertext (o el IV original para el primer bloque).

    Args:
        -

    Returns:
        tuple[str, bytearray]: Una tupla con el texto plano decodificado en
            UTF-8 y su representación original en bytes antes de eliminar el
            padding. Si el texto no puede decodificarse, la función retorna la
            excepción generada.
    """
    # Llamar a la función get_ciphertext()
    res_ciphertext = get_ciphertext()['ct']

    # Separar los bloques del ciphertext (cada 16 bytes o 32 caracteres en hex)
    blocks = [res_ciphertext[i:i+32] for i in range(0, len(res_ciphertext), 32)]

    # Variable para guardar los resultados del texto plano recuperado
    plaintext_bytes = bytearray()

    # Iterar sobre cada bloque
    for i, b_i in enumerate(blocks):
        # Avanzar al siguiente bloque si es el IV original (bloque 0)
        if i == 0:
            continue

        # Preprarar variables para guardar los estados
        plaintext = bytearray(BLOCK_SIZE)
        intermediate = bytearray(BLOCK_SIZE)
        prev_block = bytes.fromhex(blocks[i-1])

        # Iterar hacia ATRÁS en el índice del byte (desde el 15 hasta el 0)
        for byte_index in range(BLOCK_SIZE - 1, -1, -1):
            # Definir el byte objetivo y el padding a lograr
            pad_obj = BLOCK_SIZE - byte_index

            # Crear el IV falso que hará la fuerza bruta
            iv_fake = bytearray(BLOCK_SIZE)

            # Modificar el IV falso con los bits que conocemos
            for j in range(byte_index + 1, BLOCK_SIZE):
                iv_fake[j] = intermediate[j] ^ pad_obj

            # Probar los 256 valores hasta un padding válido
            for byte_try in range(256):
                # Construir el IV falso
                iv_fake[byte_index] = byte_try

                # Construir el payload
                payload = bytes(iv_fake) + bytes.fromhex(b_i)

                # Verificar el padding con la función
                response_padding = verificar_relleno_oracle(
                    payload.hex())['result']

                # El padding es correcto
                if response_padding:
                    # Mitigar errores por coincidencia


                    # Modificar el byte anterior
                    if byte_index > 0:
                        iv_fake[byte_index - 1] ^= 1

                    # Construir el nuevo payload
                    payload_verification = bytes(iv_fake) + bytes.fromhex(b_i)

                    # Reenviar otra petición para verificar
                    res_verification = verificar_relleno_oracle(
                        payload_verification.hex())['result']

                    # Verificar si el mensaje era igual al padding
                    if not res_verification:
                        # Restaurar el IV fake dadas propiedades del XOR
                        iv_fake[byte_index - 1] ^= 1
                        continue

                    # Operaciones matemáticas (Si el padding era correcto)
                    intermediate[byte_index] = iv_fake[byte_index] ^ pad_obj

                    # Recuperar el carácter del texto plano
                    plaintext[byte_index] = intermediate[byte_index] ^ \
                                            prev_block[byte_index]

                    # Asumimos que siempre tendremos exito
                    break

        # Actualizar las listas
        plaintext_bytes += plaintext

    # Limpiar los resultados a lenguaje
    try:
        # Limpiamos el padding
        plaintext_str = unpad(plaintext_bytes, BLOCK_SIZE)

        # Bytes a UTF-8
        result = plaintext_str.decode("utf-8")

        result_raw = plaintext_bytes
        return result, result_raw
    except Exception as e:
        return "Error", e

In [ ]:
# Llamar a la función del ataque
message_hex, message_raw = padding_oracle_attack()

# Imprimir los resultados
print(f"Mensaje recuperado a través del ataque:")
print(message_raw)
print(f"\nMensaje limpiado en formato hex:")
print(message_hex)

print(f"\nVariable secreta generada aleatoriamente")
print(_mensaje_secreto)

Mensaje recuperado a través del ataque:
bytearray(b'60243514b0abbb5304e44c5836a50f1a\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10')

Mensaje limpiado en formato hex:
60243514b0abbb5304e44c5836a50f1a

Variable secreta generada aleatoriamente
60243514b0abbb5304e44c5836a50f1a


## Conclusión

Este ejercicio demuestra que la seguridad de un sistema criptográfico no depende únicamente del algoritmo de cifrado utilizado, sino también de cómo se manejan los errores durante el descifrado. Aunque AES-CBC continúa siendo criptográficamente seguro, un servidor que revela si el padding es válido o inválido actúa como un Padding Oracle.



# Aprendizaje

Tal como dice usted en clase con la cita de Shamir:
> Cryptography is typically bypassed, not penetrated

Las vulnerabilidades más graves suelen encontrarse en el diseño e implementación del sistema, y no en el algoritmo criptográfico en sí.

**- Problema 2 y 3:** AES-CBC sigue siendo seguro, pero reutilizar un IV fijo, o usar el IV como llave, filtra información.

**- Problema 4:** Triple DES no se rompe; se eligen deliberadamente llaves débiles para que el esquema colapse.

**- Problema 5:** AES-CBC no es vulnerado criptográficamente; el servidor revela información mediante un Padding Oracle.